In [1]:
import torch
assert torch.cuda.is_available(), "Select a CUDA runtime before model training"
print("CUDA GPU:", torch.cuda.get_device_name(0), "| torch:", torch.__version__)

CUDA GPU: NVIDIA GeForce RTX 5080 Laptop GPU | torch: 2.14.1+cu130


In [2]:
import json, os, random, re, time
from collections import Counter
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
try:
    display
except NameError:
    display = print

DRIVE_BASE = Path(".")
DATA_DIR = (DRIVE_BASE / "processed_edges")
CFG = {
    "SPLIT": "sai_files",  # preserve original notebook's split-aware evaluation
    "SPLIT_DIR": str(DATA_DIR),
    "TRAIN_FILE": str(DATA_DIR / "train_edges.parquet"),
    "VAL_FILE": str(DATA_DIR / "validation_edges.parquet"),
    "TEST_FILE": str(DATA_DIR / "test_edges.parquet"),
    "SYNTHETIC_TEST_FILE": str(DATA_DIR / "synthetic_test_edges.parquet"),
    "SOURCE_FILE": str(DRIVE_BASE / "blocked_cv" / "sai_edges_5min_capture.parquet"),
    "OUT_DIR": str(DRIVE_BASE / "mistral_multi_lora"),
    "HOLD_OUT": "UWF-ZeekDataSum25-1",
    "PILOT": False,
    "SEED": 42,
    "TRAIN_CAP_PER_COMBO": 500,
    "NORMAL_PER_ATTACK": 1.0,
    "VAL_CAP_PER_COMBO": 50,
    "VAL_FRACTION": 0.1,
    "TEST_CAP_PER_COMBO": 1000,
    "TEST_NORMAL": 5000,
    "BASE_MODEL": "mistralai/Mistral-7B-Instruct-v0.3",
    "MAX_SEQ_LEN": 1024,
    "LORA_RANK": 16,
    "EPOCHS": 1,
    "LEARNING_RATE": 2e-4,
    "BATCH_SIZE": 1,  # fits 16 GB more reliably than the original value of 8
    "GRAD_ACCUM": 8,
    "INFER_BATCH": 4,
    "RUN_MODEL": True,
}
CFG.update(json.loads(os.environ.get("UWF_LLM_OVERRIDES", "{}")))
if CFG["PILOT"]:
    CFG.update({"TRAIN_CAP_PER_COMBO": 150, "VAL_CAP_PER_COMBO": 20,
                "TEST_CAP_PER_COMBO": 150, "TEST_NORMAL": 600, "EPOCHS": 1})
RUN_NAME = "sai_split" if CFG["SPLIT"] == "sai_files" else f"{CFG['SPLIT']}__{CFG['HOLD_OUT']}"
OUT_DIR = Path(CFG["OUT_DIR"]) / f"{RUN_NAME}{'__pilot' if CFG['PILOT'] else ''}"
OUT_DIR.mkdir(parents=True, exist_ok=True)
random.seed(CFG["SEED"]); np.random.seed(CFG["SEED"])
print("data dir:", DATA_DIR, "| output:", OUT_DIR)

data dir: processed_edges | output: mistral_multi_lora/sai_split


In [3]:
import ast


def load_edges():
    if CFG["SPLIT"] != "sai_files":
        return pd.read_parquet(CFG["SOURCE_FILE"])

    # Different test regimes must retain separate split tags.
    files = {
        "train": CFG["TRAIN_FILE"],
        "validation": CFG["VAL_FILE"],
        "test_related": CFG["TEST_FILE"],
        "test_unseen": CFG["SYNTHETIC_TEST_FILE"],
    }
    parts = []
    required = {"train", "validation"}
    for split, raw_path in files.items():
        path = Path(raw_path) if raw_path else None
        if path is None or not path.exists():
            if split in required:
                raise FileNotFoundError(f"Missing {split} split: {raw_path!r}")
            print(f"Skipping {split} evaluation because file is unavailable: {raw_path}")
            continue
        parts.append(pd.read_parquet(path).assign(split=split))
    return pd.concat(parts, ignore_index=True)


edges = load_edges()
edges["window_start"] = pd.to_datetime(edges["window_start"])
counts = edges["connection_count"].to_numpy(dtype=float)
assert (counts >= 1).all() and np.allclose(counts, np.round(counts)), \
    "connection counts are not whole numbers: these features look log-transformed or scaled, use the files saved before log1p"


def as_list(value):
    if isinstance(value, str):
        value = ast.literal_eval(value) if value.strip().startswith("[") else [value] if value.strip() else []
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return []
    return sorted(str(v) for v in value)


edges["tactics"] = edges["tactic_targets"].map(as_list)
edges["techniques"] = edges["mitre_targets"].map(as_list)
edges["attack"] = edges["binary_target"].astype(int) == 1
edges["year_group"] = edges["window_start"].dt.year.map({2021: "2021-22", 2022: "2021-22", 2024: "2024", 2025: "2025"})

## what the source and destination did in the same window (the context the GNN gets from its graph)
window_key = [c for c in ["capture", "split"] if c in edges] + ["window_start"]
src_stats = edges.groupby(window_key + ["src_ip_zeek"]).agg(
    src_targets=("dest_ip_zeek", "nunique"), src_connections=("connection_count", "sum"),
    src_ports=("unique_destination_ports", "sum"))
dst_stats = edges.groupby(window_key + ["dest_ip_zeek"]).agg(dst_sources=("src_ip_zeek", "nunique"))
edges = edges.join(src_stats, on=window_key + ["src_ip_zeek"]).join(dst_stats, on=window_key + ["dest_ip_zeek"])

CONN_STATES = {
    "S0": "no reply to the connection attempt", "REJ": "connection rejected", "SF": "normal open and close",
    "RSTO": "reset by the originator", "RSTR": "reset by the responder", "RSTOS0": "originator sent SYN then reset",
    "RSTRH": "responder sent SYN-ACK then reset", "SH": "originator sent SYN then FIN", "SHR": "responder sent SYN-ACK then FIN",
    "S1": "established, not closed", "S2": "established, originator tried to close", "S3": "established, responder tried to close",
    "OTH": "no handshake seen",
}


def number(value):
    value = float(value or 0)
    return f"{value:,.0f}" if value >= 10 or value == int(value) else f"{value:.2f}"


def connection_text(row) -> str:
    service = row["most_common_service"] if isinstance(row["most_common_service"], str) and row["most_common_service"] else "unknown"
    state = row["most_common_connection_state"] if isinstance(row["most_common_connection_state"], str) else "unknown"
    port = row["most_common_destination_port"]
    port = "unknown" if port is None or (isinstance(port, float) and np.isnan(port)) else f"{int(port)}"
    return "\n".join([
        "Connection summary for one source-destination pair over 5 minutes:",
        f"- connections: {number(row['connection_count'])} (TCP {number(row['tcp_count'])}, UDP {number(row['udp_count'])}, ICMP {number(row['icmp_count'])})",
        f"- bytes sent: {number(row['total_orig_bytes'])}, bytes received: {number(row['total_resp_bytes'])}",
        f"- packets sent: {number(row['total_orig_packets'])}, packets received: {number(row['total_resp_packets'])}",
        f"- IP bytes sent: {number(row['total_orig_ip_bytes'])}, received: {number(row['total_resp_ip_bytes'])}, missed bytes: {number(row['total_missed_bytes'])}",
        f"- duration: total {number(row['total_duration'])} s, average {number(row['average_duration'])} s, longest {number(row['maximum_duration'])} s",
        f"- most common destination port: {port}; distinct destination ports: {number(row['unique_destination_ports'])}",
        f"- most common service: {service}",
        f"- most common connection state: {state} ({CONN_STATES.get(state, 'unknown state')})",
        f"- same 5 minutes, source overall: {number(row['src_targets'])} hosts contacted, {number(row['src_connections'])} connections",
        f"- same 5 minutes, destination overall: contacted by {number(row['dst_sources'])} sources",
    ])


def answer_json(row) -> str:
    return json.dumps({"attack": bool(row["attack"]), "tactics": row["tactics"], "techniques": row["techniques"]})


edges["text"] = [connection_text(r) for r in edges.to_dict("records")]
edges["answer"] = [answer_json(r) for r in edges.to_dict("records")]
edges["combo"] = np.where(edges["attack"],
                          [("T:" + ",".join(t)) if t else ("tactic-only:" + ",".join(a)) for t, a in zip(edges.techniques, edges.tactics)],
                          "normal")
print(f"{len(edges):,} connections | attacks: {int(edges.attack.sum()):,}",
      edges.groupby("split").attack.agg(["size", "sum"]).to_dict("index") if "split" in edges else f"| datasets: {edges.capture.nunique()}")
print("\nExample attack:\n" + edges.loc[edges.attack].iloc[0]["text"] + "\n→ " + edges.loc[edges.attack].iloc[0]["answer"])
print("\nExample normal:\n" + edges.loc[~edges.attack].iloc[0]["text"] + "\n→ " + edges.loc[~edges.attack].iloc[0]["answer"])

678,948 connections | attacks: 54,777 {'test_related': {'size': 69161, 'sum': 5670}, 'test_unseen': {'size': 894, 'sum': 127}, 'train': {'size': 541763, 'sum': 43367}, 'validation': {'size': 67130, 'sum': 5613}}

Example attack:
Connection summary for one source-destination pair over 5 minutes:
- connections: 1 (TCP 1, UDP 0, ICMP 0)
- bytes sent: 0, bytes received: 0
- packets sent: 6, packets received: 3
- IP bytes sent: 252, received: 132, missed bytes: 0
- duration: total 0.00 s, average 0.00 s, longest 0.00 s
- most common destination port: 53; distinct destination ports: 1
- most common service: unknown
- most common connection state: RSTO (reset by the originator)
- same 5 minutes, source overall: 4 hosts contacted, 5 connections
- same 5 minutes, destination overall: contacted by 1 sources
→ {"attack": true, "tactics": ["Reconnaissance"], "techniques": ["T1595"]}

Example normal:
Connection summary for one source-destination pair over 5 minutes:
- connections: 10 (TCP 0, UDP 10

In [4]:
from sklearn.model_selection import train_test_split


def split_masks(df):
    """Returns (train_pool, test) boolean masks for the chosen SPLIT."""
    if CFG["SPLIT"] == "sai_files":
        return (df.split == "train").to_numpy(), (df.split == "test_related").to_numpy()
    if CFG["SPLIT"] == "leave_one_dataset_out":
        test = (df.capture == CFG["HOLD_OUT"]).to_numpy()
    elif CFG["SPLIT"] == "year":
        test = (df.year_group == CFG["HOLD_OUT"]).to_numpy()
    elif CFG["SPLIT"] == "random_window":
        ## Sai's split: whole 5-minute windows shuffled 80/10/10, stratified by whether the window has an attack
        windows = df.groupby("window_start").attack.max().reset_index()
        train_w, rest_w = train_test_split(windows, test_size=0.2, random_state=CFG["SEED"], stratify=windows.attack)
        _, test_w = train_test_split(rest_w, test_size=0.5, random_state=CFG["SEED"], stratify=rest_w.attack)
        test = df.window_start.isin(set(test_w.window_start)).to_numpy()
        return df.window_start.isin(set(train_w.window_start)).to_numpy(), test
    else:
        raise ValueError(CFG["SPLIT"])
    return ~test, test


train_pool_mask, test_mask = split_masks(edges)
HAS_TEST = bool(test_mask.any())
assert HAS_TEST or CFG["SPLIT"] == "sai_files", f"nothing to test: check HOLD_OUT={CFG['HOLD_OUT']!r}"
pool, test_pool = edges[train_pool_mask], edges[test_mask]


def capped_sample(frame, cap):
    """Up to `cap` connections per attack combination, the same number of normal ones, duplicate texts removed
    (keeping the majority answer), shuffled."""
    parts = [g.sample(min(len(g), cap), random_state=CFG["SEED"]) for _, g in frame[frame.attack].groupby("combo")]
    attacks = pd.concat(parts) if parts else frame.iloc[0:0]
    normal = frame[~frame.attack]
    normals = normal.sample(min(len(normal), int(len(attacks) * CFG["NORMAL_PER_ATTACK"]) or 1), random_state=CFG["SEED"])
    sample = pd.concat([attacks, normals])
    majority = sample.groupby("text").answer.agg(lambda s: s.value_counts().index[0])
    sample = sample.drop_duplicates("text").assign(answer=lambda d: d.text.map(majority)).copy()
    # Keep row-level targets consistent with the majority answer used for training.
    labels = sample["answer"].map(json.loads)
    sample["attack"] = labels.map(lambda d: bool(d["attack"]))
    sample["tactics"] = labels.map(lambda d: d["tactics"])
    sample["techniques"] = labels.map(lambda d: d["techniques"])
    return sample.sample(frac=1.0, random_state=CFG["SEED"])


if CFG["SPLIT"] == "sai_files":
    ## training from Sai's training file, validation from her validation file (texts already in training dropped)
    train_set = capped_sample(pool, CFG["TRAIN_CAP_PER_COMBO"])
    val_set = capped_sample(edges[edges.split == "validation"], CFG["VAL_CAP_PER_COMBO"])
    val_set = val_set[~val_set.text.isin(set(train_set.text))]
else:
    train_all = capped_sample(pool, CFG["TRAIN_CAP_PER_COMBO"])
    n_val = max(1, int(len(train_all) * CFG["VAL_FRACTION"]))
    val_set, train_set = train_all.iloc[:n_val], train_all.iloc[n_val:]

# Evaluate both test regimes separately with the same original stratum-based weighting.
# Sample weights are defined within each dataset, never across the two datasets.
def weighted_test_sample(frame):
    if frame.empty:
        return frame.assign(weight=pd.Series(dtype=float))
    parts = []
    for combo, group in frame.groupby("combo"):
        cap = CFG["TEST_NORMAL"] if combo == "normal" else CFG["TEST_CAP_PER_COMBO"]
        subset = group.sample(min(len(group), cap), random_state=CFG["SEED"])
        parts.append(subset.assign(weight=len(group) / len(subset)))
    return pd.concat(parts) if parts else frame.assign(weight=1.0)

# For alternative split modes, the original held-out test remains the related test.
test_pools = {"related_sessions": test_pool}
if CFG["SPLIT"] == "sai_files":
    test_pools["unseen_synthetic"] = edges.loc[edges["split"] == "test_unseen"]

test_sets = {name: weighted_test_sample(raw) for name, raw in test_pools.items() if len(raw)}
HAS_TEST = bool(test_sets)

# Keep a legacy alias for any interactive code that expects test_set.
test_set = test_sets.get("related_sessions", edges.iloc[:0].assign(weight=pd.Series(dtype=float)))

# Techniques seen by the trainer are the only ones considered learnable.
learnable = {t for ts in pool.loc[pool.attack, "techniques"] for t in ts}

summary_frames = {"train": train_set, "validation": val_set, **test_sets}
summary = pd.DataFrame.from_dict({
    name: {
        "connections": len(frame),
        "attacks": int(frame.attack.sum()),
        "represents (weighted)": int(round(frame.weight.sum())) if "weight" in frame else len(frame),
    }
    for name, frame in summary_frames.items()
}, orient="index")
display(summary)

train_texts = set(train_set.text)
train_techniques = {t for ts in pool.loc[pool.attack, "techniques"] for t in ts}
for name, frame in test_sets.items():
    heldout_techniques = {t for ts in frame.techniques for t in ts}
    print(name,
          "| exact training-text overlap:", f"{frame.text.isin(train_texts).mean():.2%}",
          "| MITRE labels present in training:", len(heldout_techniques & train_techniques),
          "| labels absent from training:", sorted(heldout_techniques - train_techniques))

for name, frame in summary_frames.items():
    frame.drop(columns=["tactic_targets", "mitre_targets"], errors="ignore").to_parquet(
        OUT_DIR / f"{name}.parquet", index=False
    )

,connections,attacks,represents (weighted)
train,4998,2295,4998
validation,475,249,475
related_sessions,7882,2882,69161
unseen_synthetic,894,127,894


related_sessions | exact training-text overlap: 53.95% | MITRE labels present in training: 16 | labels absent from training: []
unseen_synthetic | exact training-text overlap: 0.00% | MITRE labels present in training: 27 | labels absent from training: []


In [5]:
## compare the synthetic test file with Sai's training file
feats = ["connection_count", "total_orig_bytes", "total_resp_bytes", "total_duration",
         "unique_destination_ports", "src_targets", "src_connections", "dst_sources"]
display(edges[edges.split.isin(["train", "test"])].groupby(["split", "attack"])[feats].median().T)

for split in ["train", "test"]:
    e = edges[edges.split == split]
    w = e.groupby("window_start").attack.agg(["min", "max"])
    mixed = w[(w["min"] == 0) & (w["max"] == 1)].index
    print(f"{split}: rows {len(e):,} | windows {e.window_start.nunique():,} | attack rate {e.attack.mean():.3f}"
          f" | attack rows in windows that also have normal traffic {e[e.attack & e.window_start.isin(mixed)].shape[0] / max(e.attack.sum(), 1):.3f}")
    print("   top services:", e.most_common_service.value_counts().head(5).to_dict())
print("test texts identical to a training text:", round(test_set.text.isin(set(train_set.text)).mean(), 3))

split                          train          
attack                         False     True 
connection_count            3.000000  2.000000
total_orig_bytes          215.000000  0.000000
total_resp_bytes            0.000000  0.000000
total_duration              0.000354  0.002411
unique_destination_ports    1.000000  1.000000
src_targets                 1.000000  4.000000
src_connections             8.000000  8.000000
dst_sources                 2.000000  1.000000

train: rows 541,763 | windows 20,149 | attack rate 0.080 | attack rows in windows that also have normal traffic 0.007
   top services: {'dns': 434004, 'unknown': 84480, 'ntp': 6884, 'ssl': 5791, 'dhcp': 5715}
test: rows 0 | windows 0 | attack rate nan | attack rows in windows that also have normal traffic 0.000
   top services: {}
test texts identical to a training text: 0.539


In [6]:
# Define allowable output labels only from training data, not either test split.
label_rows = edges.loc[edges.split == "train"] if "split" in edges else pool
TACTICS = sorted(set(t for ts in label_rows.tactics for t in ts))
TECHNIQUES = sorted(set(t for ts in label_rows.techniques for t in ts))
TECHNIQUE_NAMES = {
    "T1016": "System Network Configuration Discovery", "T1018": "Remote System Discovery", "T1046": "Network Service Discovery",
    "T1048": "Exfiltration Over Alternative Protocol", "T1059": "Command and Scripting Interpreter", "T1071": "Application Layer Protocol",
    "T1078": "Valid Accounts", "T1110": "Brute Force", "T1112": "Modify Registry", "T1133": "External Remote Services",
    "T1136": "Create Account", "T1190": "Exploit Public-Facing Application", "T1203": "Exploitation for Client Execution",
    "T1204": "User Execution", "T1210": "Exploitation of Remote Services", "T1505": "Server Software Component",
    "T1546": "Event Triggered Execution", "T1547": "Boot or Logon Autostart Execution", "T1548": "Abuse Elevation Control Mechanism",
    "T1557": "Adversary-in-the-Middle", "T1566": "Phishing", "T1571": "Non-Standard Port", "T1587": "Develop Capabilities",
    "T1589": "Gather Victim Identity Information", "T1590": "Gather Victim Network Information",
    "T1592": "Gather Victim Host Information", "T1595": "Active Scanning",
}
TASKS = ("attack_detection", "tactic_classification", "technique_classification")
TASK_SYSTEMS = {
    "attack_detection": (
        "You are a network security analyst. Given a five-minute Zeek connection summary, "
        "classify whether it is part of an attack. Answer only JSON with exactly one key: "
        "{\"attack\": true} or {\"attack\": false}."
    ),
    "tactic_classification": (
        "You are a network security analyst. Given a five-minute Zeek connection summary, "
        "predict all applicable MITRE ATT&CK tactic labels. Normal traffic has none. "
        "Answer JSON only: {\"tactics\": [\"...\"]}. Allowed labels: " + ", ".join(TACTICS)
    ),
    "technique_classification": (
        "You are a network security analyst. Given a five-minute Zeek connection summary, "
        "predict all applicable MITRE ATT&CK technique IDs. Normal traffic has none. "
        "Answer JSON only: {\"techniques\": [\"T1046\"]}. Allowed IDs: " +
        "; ".join(f"{t} {TECHNIQUE_NAMES.get(t, '')}".strip() for t in TECHNIQUES)
    ),
}

def task_answer(task, row):
    if task == "attack_detection":
        return json.dumps({"attack": bool(row["attack"])})
    if task == "tactic_classification":
        return json.dumps({"tactics": row["tactics"]})
    if task == "technique_classification":
        return json.dumps({"techniques": row["techniques"]})
    raise ValueError(task)

def task_messages(task, text):
    return [
        {"role": "system", "content": TASK_SYSTEMS[task]},
        {"role": "user", "content": text},
    ]

def parse_answer(raw: str) -> dict:
    match = re.search(r"\{.*?\}", raw or "", flags=re.S)
    result = {"attack": None, "tactics": [], "techniques": [], "invalid": [], "parsed": False}
    if not match:
        return result
    try:
        payload = json.loads(match.group(0))
    except json.JSONDecodeError:
        return result
    if not isinstance(payload, dict):
        return result
    result["parsed"] = True
    result["attack"] = payload.get("attack") if isinstance(payload.get("attack"), bool) else None
    for key, allowed in (("tactics", TACTICS), ("techniques", TECHNIQUES)):
        values = payload.get(key) or []
        if not isinstance(values, list):
            result["invalid"].append(str(values)); continue
        result[key] = sorted({str(v) for v in values if str(v) in allowed})
        result["invalid"] += [str(v) for v in values if str(v) not in allowed]
    return result

print("Classes:", len(TACTICS), "tactics and", len(TECHNIQUES), "techniques")

Classes: 13 tactics and 27 techniques


In [7]:
from sklearn.metrics import average_precision_score, f1_score, roc_auc_score
import numpy as np
import pandas as pd

def score_predictions(pred: pd.DataFrame, frame: pd.DataFrame) -> dict:
    """pred: index aligned with frame; columns p_attack, attack_pred, tactics_pred, techniques_pred, parsed, invalid."""
    w = frame["weight"].to_numpy() if "weight" in frame else np.ones(len(frame))
    y = frame["attack"].to_numpy().astype(int)
    p = pred["p_attack"].to_numpy()
    yhat = pred["attack_pred"].fillna(False).to_numpy().astype(bool)
    tp, fp = float(w[yhat & (y == 1)].sum()), float(w[yhat & (y == 0)].sum())
    fn, tn = float(w[~yhat & (y == 1)].sum()), float(w[~yhat & (y == 0)].sum())
    scores = {
        "parsed_answers": float(pred["parsed"].mean()),
        "answers_with_invalid_labels": float((pred["invalid"].map(len) > 0).mean()),
        "attack_pr_auc": float(average_precision_score(y, p, sample_weight=w)) if 0 < y.sum() < len(y) else np.nan,
        "attack_roc_auc": float(roc_auc_score(y, p, sample_weight=w)) if 0 < y.sum() < len(y) else np.nan,
        "attack_recall": tp / max(tp + fn, 1e-9), "attack_precision": tp / max(tp + fp, 1e-9),
        "false_alarm_rate": fp / max(fp + tn, 1e-9),
    }
    attacks = frame["attack"].to_numpy()
    for key, classes in [("tactics", TACTICS), ("techniques", TECHNIQUES)]:
        true_lists = [list(x) if isinstance(x, (list, np.ndarray, set)) else [] for x in frame.loc[attacks, key]]
        pred_lists = [list(x) if isinstance(x, (list, np.ndarray, set)) else [] for x in pred.loc[attacks, f"{key}_pred"]]
        keep = [i for i, t in enumerate(true_lists) if t]
        if not keep:
            continue
        # Include held-out-only labels when scoring. Missing such labels count as false negatives.
        eval_classes = sorted(set(classes) | {c for true in true_lists for c in true})
        index = {c: i for i, c in enumerate(eval_classes)}
        Y = np.zeros((len(keep), len(eval_classes))); P = np.zeros_like(Y)
        for row, i in enumerate(keep):
            for c in true_lists[i]: Y[row, index[c]] = 1
            for c in pred_lists[i]:
                if c in index: P[row, index[c]] = 1
        present = Y.sum(0) > 0
        scores[f"{key}_exact_match"] = float(np.mean([set(true_lists[i]) == set(pred_lists[i]) for i in keep]))
        scores[f"{key}_first_correct"] = float(np.mean([len(pred_lists[i]) > 0 and pred_lists[i][0] in true_lists[i] for i in keep]))
        scores[f"{key}_micro_f1"] = float(f1_score(Y, P, average="micro", zero_division=0))
        scores[f"{key}_macro_f1"] = float(f1_score(Y[:, present], P[:, present], average="macro", zero_division=0))
    return scores

In [ ]:
%pip install -q -U peft trl

In [ ]:
%pip install -q -U bitsandbytes>=0.46.1

In [8]:
import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset
from trl import SFTConfig, SFTTrainer
from tqdm.auto import tqdm

DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
QUANT = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                           bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=DTYPE)
tokenizer = AutoTokenizer.from_pretrained(CFG["BASE_MODEL"])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def load_base():
    base = AutoModelForCausalLM.from_pretrained(
        CFG["BASE_MODEL"], quantization_config=QUANT,
        device_map={"": 0}, dtype=DTYPE)
    base.config.use_cache = False
    return base

model = load_base() if CFG["RUN_MODEL"] else None

def infer_task(task, frame):
    """Greedy generation. The binary task also estimates P(attack) from first-token logits."""
    tokenizer.padding_side = "left"
    result = {}
    texts = list(dict.fromkeys(frame["text"].tolist()))
    prefix = '{"attack":' if task == "attack_detection" else ""
    true_ids = tokenizer.encode(" true", add_special_tokens=False)
    false_ids = tokenizer.encode(" false", add_special_tokens=False)
    if task == "attack_detection" and (len(true_ids) != 1 or len(false_ids) != 1):
        raise RuntimeError("Mistral tokenizer did not return single-token attack labels. Implement full candidate log-prob scoring before claiming PR-AUC/ROC-AUC.")
    for start in tqdm(
            range(0, len(texts), CFG["INFER_BATCH"]),
            desc=f"Inference: {task}",
            unit="batch"
        ):
        batch = texts[start:start + CFG["INFER_BATCH"]]
        prompts = [tokenizer.apply_chat_template(task_messages(task, t), tokenize=False,
                                                 add_generation_prompt=True) + prefix for t in batch]
        inputs = tokenizer(
                            prompts,
                            return_tensors="pt",
                            padding=True,
                            truncation=True,
                            max_length=CFG["MAX_SEQ_LEN"],
                            add_special_tokens=False
                        ).to(model.device)
        with torch.inference_mode():
            gen = model.generate(
                **inputs,
                max_new_tokens=24 if prefix else 96,
                do_sample=False,
                use_cache=True,
                return_dict_in_generate=True,
                output_scores=bool(prefix),
                pad_token_id=tokenizer.pad_token_id,
            )
        tokens = gen.sequences[:, inputs["input_ids"].shape[1]:]
        raws = tokenizer.batch_decode(tokens, skip_special_tokens=True)
        probs = (torch.softmax(gen.scores[0][:, [true_ids[0], false_ids[0]]].float(), dim=-1)[:, 0]
                 .cpu().tolist()) if prefix else [float("nan")] * len(batch)
        for t, raw, prob in zip(batch, raws, probs):
            result[t] = (parse_answer(prefix + raw), float(prob), prefix + raw)
    return result

def run_model(frame, label, dataset_name="related_sessions"):
    """Use three independent task prompts to construct original notebook's unified score schema."""
    path = OUT_DIR / f"predictions_{dataset_name}_{label}.parquet"
    if path.exists():
        cached = pd.read_parquet(path).set_index("row")
        if cached.index.tolist() == frame.index.tolist():
            return cached
        path.unlink()
    model.eval()
    model.config.use_cache = True
    model.generation_config.use_cache = True
    predictions = {}
    for task in TASKS:
        if label == "fine_tuned":
            model.set_adapter(task)
        predictions[task] = infer_task(task, frame)
    records = []
    for idx, text in zip(frame.index, frame.text):
        a, p, ra = predictions["attack_detection"][text]
        t, _, rt = predictions["tactic_classification"][text]
        m, _, rm = predictions["technique_classification"][text]
        records.append({"row": idx, "p_attack": p,
                        "attack_pred": a["attack"] if a["attack"] is not None else p >= 0.5,
                        "tactics_pred": t["tactics"], "techniques_pred": m["techniques"],
                        "parsed": bool(a["parsed"] and t["parsed"] and m["parsed"]),
                        "invalid": a["invalid"] + t["invalid"] + m["invalid"],
                        "raw": str({"attack": ra, "tactics": rt, "techniques": rm})})
    pred = pd.DataFrame(records).set_index("row")
    pred.reset_index().to_parquet(path, index=False)
    return pred

print("Loaded base:", CFG["BASE_MODEL"])

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loaded base: mistralai/Mistral-7B-Instruct-v0.3


In [9]:
model.eval()
model.config.use_cache = True
model.generation_config.use_cache = True

In [9]:
import time
import torch
import pandas as pd

# Benchmark inference on 64 connections
sample = test_sets["related_sessions"].sample(
    n=min(64, len(test_sets["related_sessions"])),
    random_state=42
)

model.eval()
model.config.use_cache = True
model.generation_config.use_cache = True

# GPU warmup
infer_task("attack_detection", sample.iloc[:4])

task_times = {}

for task in TASKS:
    torch.cuda.synchronize()
    start = time.perf_counter()

    infer_task(task, sample)

    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start

    task_times[task] = elapsed

    print(f"{task}: {elapsed:.2f}s")

# Calculate estimated runtime
seconds_per_connection = sum(task_times.values()) / len(sample)

print("\nEstimated full evaluation:")

total_seconds = 0

for name, frame in test_sets.items():
    # infer_task deduplicates identical input texts
    n_unique = frame["text"].nunique()
    estimated_seconds = seconds_per_connection * n_unique
    total_seconds += estimated_seconds

    print(
        f"{name}: {n_unique:,} unique inputs, "
        f"{estimated_seconds / 3600:.2f} hours"
    )

print(f"\nTotal: {total_seconds / 3600:.2f} hours")


Inference: attack_detection:   0%|          | 0/1 [00:00<?, ?batch/s]

Inference: attack_detection:   0%|          | 0/16 [00:00<?, ?batch/s]

attack_detection: 5.04s


Inference: tactic_classification:   0%|          | 0/16 [00:00<?, ?batch/s]

tactic_classification: 24.65s


Inference: technique_classification:   0%|          | 0/16 [00:00<?, ?batch/s]

technique_classification: 46.72s

Estimated full evaluation:
related_sessions: 4,127 unique inputs, 1.37 hours
unseen_synthetic: 894 unique inputs, 0.30 hours

Total: 1.67 hours


In [10]:
zero_scores_by_split = {}
zero_preds_by_split = {}

if CFG["RUN_MODEL"] and HAS_TEST:

    for dataset_name, frame in test_sets.items():

        print(f"\n{'=' * 60}")
        print(f"Zero-shot evaluation: {dataset_name}")
        print(f"Connections: {len(frame):,}")
        print(f"Unique inputs: {frame['text'].nunique():,}")
        print(f"{'=' * 60}")

        torch.cuda.synchronize()
        start = time.perf_counter()

        pred = run_model(
            frame,
            "zero_shot",
            dataset_name=dataset_name
        )

        scores = score_predictions(pred, frame)

        zero_preds_by_split[dataset_name] = pred
        zero_scores_by_split[dataset_name] = scores

        with open(
            OUT_DIR / f"scores_{dataset_name}_zero_shot.json",
            "w"
        ) as f:
            json.dump(scores, f, indent=2)

        torch.cuda.synchronize()
        elapsed = time.perf_counter() - start

        print(f"\nCompleted: {dataset_name}")
        print(f"Elapsed: {elapsed / 60:.2f} minutes")

        display(
            pd.Series(
                scores,
                name=f"{dataset_name} | zero-shot"
            ).round(4)
        )

else:
    print("Skipping zero-shot scoring")



Zero-shot evaluation: related_sessions
Connections: 7,882
Unique inputs: 4,127


Inference: attack_detection:   0%|          | 0/1032 [00:00<?, ?batch/s]

Inference: tactic_classification:   0%|          | 0/1032 [00:00<?, ?batch/s]

Inference: technique_classification:   0%|          | 0/1032 [00:00<?, ?batch/s]


Completed: related_sessions
Elapsed: 78.82 minutes


parsed_answers                 1.0000
answers_with_invalid_labels    0.1290
attack_pr_auc                  0.0692
attack_roc_auc                 0.4321
attack_recall                  0.0012
attack_precision               0.0050
false_alarm_rate               0.0220
tactics_exact_match            0.0000
tactics_first_correct          0.0014
tactics_micro_f1               0.3023
tactics_macro_f1               0.0704
techniques_exact_match         0.0004
techniques_first_correct       0.0004
techniques_micro_f1            0.0004
techniques_macro_f1            0.0000
Name: related_sessions | zero-shot, dtype: float64

... (2 outputs left out)

Inference: tactic_classification:   0%|          | 0/224 [00:00<?, ?batch/s]

Inference: technique_classification:   0%|          | 0/224 [00:00<?, ?batch/s]


Completed: unseen_synthetic
Elapsed: 21.88 minutes


parsed_answers                 1.0000
answers_with_invalid_labels    0.3680
attack_pr_auc                  0.1440
attack_roc_auc                 0.5264
attack_recall                  0.0079
attack_precision               0.0455
false_alarm_rate               0.0274
tactics_exact_match            0.0315
tactics_first_correct          0.1181
tactics_micro_f1               0.2044
tactics_macro_f1               0.1022
techniques_exact_match         0.0000
techniques_first_correct       0.0472
techniques_micro_f1            0.0331
techniques_macro_f1            0.0033
Name: unseen_synthetic | zero-shot, dtype: float64

In [11]:
import numpy as np
import pandas as pd

dataset_name = "related_sessions"

pred = zero_preds_by_split[dataset_name]
frame = test_sets[dataset_name].loc[pred.index]

w = frame["weight"].to_numpy()
actual = frame["attack"].astype(bool).to_numpy()
predicted = pred["attack_pred"].fillna(False).astype(bool).to_numpy()

print("True attack rate:", np.average(actual, weights=w))
print("Predicted attack rate:", np.average(predicted, weights=w))

print("\nAttack probability statistics:")
display(
    pd.DataFrame({
        "actual": actual,
        "p_attack": pred["p_attack"].to_numpy()
    }).groupby("actual")["p_attack"].describe()
)

print("\nExample predictions:")
display(
    pred[["attack_pred", "p_attack", "raw"]].head(10)
)


True attack rate: 0.08198262026286492
Predicted attack rate: 0.020297595465652615

Attack probability statistics:


,count,mean,std,min,25%,50%,75%,max
actual,,,,,,,,
False,5000.0,0.023928,0.091809,1.125352e-07,2.902312e-06,0.000017,0.002801,0.777300
True,2882.0,0.003679,0.041312,1.275190e-07,8.851570e-07,0.000011,0.000123,0.970688



Example predictions:


,attack_pred,p_attack,raw
row,,,
619849,True,9.706877e-01,"{'attack': '{""attack"":true}', 'tactics': '{""ta..."
619428,False,7.889263e-06,"{'attack': '{""attack"":false}', 'tactics': '{""t..."
627018,False,2.305067e-04,"{'attack': '{""attack"":false}', 'tactics': '{""t..."
630288,False,1.275190e-07,"{'attack': '{""attack"":false}', 'tactics': '{""t..."
639374,False,1.357433e-07,"{'attack': '{""attack"":false}', 'tactics': '{""t..."
676492,False,3.775407e-01,"{'attack': '{""attack"":false}', 'tactics': '{""t..."
650667,False,3.208213e-01,"{'attack': '{""attack"":false}', 'tactics': '{""t..."
664929,False,2.450850e-01,"{'attack': '{""attack"":false}', 'tactics': '{""t..."
655420,False,1.584362e-04,"{'attack': '{""attack"":false}', 'tactics': '{""t..."


In [10]:
from datasets import Dataset

TASKS = (
    "attack_detection",
    "tactic_classification",
    "technique_classification"
)

ADAPTER_ROOT = OUT_DIR / "adapters"
ADAPTER_ROOT.mkdir(parents=True, exist_ok=True)


from datasets import Dataset

def make_task_dataset(frame, task):
    examples = []

    for row in frame.to_dict("records"):

        prompt = tokenizer.apply_chat_template(
            task_messages(task, row["text"]),
            tokenize=False,
            add_generation_prompt=True
        )

        completion = task_answer(task, row)

        # Match attack_detection inference exactly
        if task == "attack_detection":
            prefix = '{"attack":'
            assert completion.startswith(prefix)

            prompt += prefix
            completion = completion[len(prefix):]

        # Tokenize prompt and completion separately
        prompt_ids = tokenizer(
            prompt,
            add_special_tokens=False
        )["input_ids"]

        completion_ids = tokenizer(
            completion,
            add_special_tokens=False
        )["input_ids"]

        completion_ids.append(tokenizer.eos_token_id)

        input_ids = prompt_ids + completion_ids

        if len(input_ids) > CFG["MAX_SEQ_LEN"]:
            raise ValueError(
                f"{task}: {len(input_ids)} tokens exceeds "
                f"MAX_SEQ_LEN={CFG['MAX_SEQ_LEN']}. "
                "Increase sequence length or shorten prompts."
            )

        labels = (
            [-100] * len(prompt_ids)
            + completion_ids
        )

        examples.append({
            "input_ids": input_ids,
            "attention_mask": [1] * len(input_ids),
            "labels": labels
        })

    return Dataset.from_list(examples)


for task in TASKS:
    ds = make_task_dataset(train_set, task)
    print(f"{task}: {len(ds):,} training examples")

attack_detection: 4,998 training examples
tactic_classification: 4,998 training examples
technique_classification: 4,998 training examples


In [11]:
import os
import gc
import math
import shutil
import inspect
import subprocess
from pathlib import Path

import torch
import pandas as pd
import trl
import transformers
import peft

from peft import (
    LoraConfig,
    PeftModel,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from trl import SFTConfig, SFTTrainer
from transformers.trainer_utils import get_last_checkpoint

# Ensure Triton can compile its CUDA support code
if shutil.which("gcc") is None or shutil.which("g++") is None:
    if os.geteuid() != 0:
        raise RuntimeError(
            "GCC/G++ missing. Install build-essential "
            "with administrator privileges."
        )

    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(
        ["apt-get", "install", "-y", "build-essential"],
        check=True,
    )

os.environ["CC"] = shutil.which("gcc")
os.environ["CXX"] = shutil.which("g++")

print("C compiler:", os.environ["CC"])
print("C++ compiler:", os.environ["CXX"])
print("TRL:", trl.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)

assert torch.cuda.is_available(), "CUDA GPU not available"
print("GPU:", torch.cuda.get_device_name(0))

# Absolute, consistent output paths
OUT_DIR = Path(OUT_DIR).expanduser().resolve()
ADAPTER_ROOT = OUT_DIR / "adapters"
ADAPTER_ROOT.mkdir(parents=True, exist_ok=True)

# Release stale objects from failed training attempts
for name in (
    "trainer",
    "model",
    "train_ds",
    "eval_ds",
    "training_args",
    "args",
):
    globals().pop(name, None)

gc.collect()
torch.cuda.empty_cache()

print("Adapter directory:", ADAPTER_ROOT)

C compiler: /usr/bin/gcc
C++ compiler: /usr/bin/g++
TRL: 1.17.1
Transformers: 5.22.0
PEFT: 0.21.2
GPU: NVIDIA GeForce RTX 5080 Laptop GPU
Adapter directory: /workspace/mistral_multi_lora/sai_split/adapters


In [12]:
task_name = "attack_detection"

ds = make_task_dataset(train_set, task_name)

example = ds[0]

prompt_length = sum(
    label == -100 for label in example["labels"]
)

target_ids = [
    label for label in example["labels"]
    if label != -100
]

print("Examples:", len(ds))
print("Prompt tokens:", prompt_length)
print("Target tokens:", len(target_ids))

print(
    "Target:",
    tokenizer.decode(
        target_ids,
        skip_special_tokens=True
    )
)

assert prompt_length > 0
assert len(target_ids) > 0
assert len(example["input_ids"]) == len(example["labels"])

Examples: 4998
Prompt tokens: 276
Target tokens: 3
Target: true}


In [13]:
def saved_adapter_is_valid(path):
    path = Path(path)

    config_ok = (path / "adapter_config.json").is_file()

    weights_ok = any(
        p.is_file() and p.stat().st_size > 0
        for p in [
            path / "adapter_model.safetensors",
            path / "adapter_model.bin",
        ]
    )

    return config_ok and weights_ok


def train_one_adapter(task_name):

    if task_name not in TASKS:
        raise ValueError(f"Unknown task: {task_name}")

    adapter_dir = ADAPTER_ROOT / task_name

    if saved_adapter_is_valid(adapter_dir):
        print(f"Already trained: {task_name}")
        return adapter_dir

    tokenizer.padding_side = "right"

    local_model = None
    local_trainer = None

    try:
        # Prepare datasets
        train_ds = make_task_dataset(train_set, task_name)
        eval_ds = (
            make_task_dataset(val_set, task_name)
            if len(val_set) else None
        )

        assert len(train_ds) > 0, "Training dataset is empty"

        batch_size = int(CFG["BATCH_SIZE"])
        accum_steps = int(CFG["GRAD_ACCUM"])

        # Approximately 3% learning-rate warmup
        total_steps = (
            math.ceil(len(train_ds) / (batch_size * accum_steps))
            * int(CFG["EPOCHS"])
        )
        warmup_steps = max(1, round(total_steps * 0.03))

        print(f"\nTraining: {task_name}")
        print("Samples:", len(train_ds))
        print("Expected optimizer steps:", total_steps)
        print("Warmup steps:", warmup_steps)

        # Load fresh quantized Mistral
        local_model = load_base()

        local_model = prepare_model_for_kbit_training(
            local_model,
            use_gradient_checkpointing=True,
            gradient_checkpointing_kwargs={
                "use_reentrant": False
            },
        )

        lora_config = LoraConfig(
            r=CFG["LORA_RANK"],
            lora_alpha=CFG["LORA_RANK"],
            lora_dropout=0.0,
            bias="none",
            task_type="CAUSAL_LM",
            target_modules=[
                "q_proj", "k_proj", "v_proj", "o_proj",
                "gate_proj", "up_proj", "down_proj",
            ],
        )

        local_model = get_peft_model(
            local_model,
            lora_config
        )

        local_model.print_trainable_parameters()

        # Training arguments
        config = dict(
            output_dir=str(OUT_DIR / "trainer" / task_name),

            max_length=CFG["MAX_SEQ_LEN"],
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=1,
            gradient_accumulation_steps=accum_steps,
            num_train_epochs=CFG["EPOCHS"],

            learning_rate=CFG["LEARNING_RATE"],
            warmup_steps=warmup_steps,
            lr_scheduler_type="linear",
            optim="paged_adamw_8bit",
            weight_decay=0.01,

            gradient_checkpointing=True,
            bf16=(DTYPE == torch.bfloat16),
            fp16=(DTYPE == torch.float16),

            # Minimal logs
            logging_strategy="steps",
            logging_steps=100,
            logging_first_step=True,
            disable_tqdm=True,
            report_to="none",

            # Recovery checkpoints
            save_strategy="steps",
            save_steps=100,
            save_total_limit=2,

            completion_only_loss=True,
            packing=False,
            seed=CFG["SEED"],
        )

        # Handle API differences explicitly
        supported = set(inspect.signature(SFTConfig).parameters)

        if "gradient_checkpointing_kwargs" in supported:
            config["gradient_checkpointing_kwargs"] = {
                "use_reentrant": False
            }

        if "eval_strategy" in supported:
            config["eval_strategy"] = (
                "steps" if eval_ds is not None else "no"
            )
        elif "evaluation_strategy" in supported:
            config["evaluation_strategy"] = (
                "steps" if eval_ds is not None else "no"
            )
        elif eval_ds is not None:
            raise RuntimeError(
                "No compatible evaluation strategy argument"
            )

        if eval_ds is not None:
            config["eval_steps"] = 200

        unsupported = set(config) - supported

        if unsupported:
            raise RuntimeError(
                f"Unsupported SFTConfig parameters: {unsupported}"
            )

        training_args = SFTConfig(**config)

        local_trainer = SFTTrainer(
            model=local_model,
            args=training_args,
            train_dataset=train_ds,
            eval_dataset=eval_ds,
            processing_class=tokenizer,
        )

        # Check labels before expensive training
        batch = next(iter(local_trainer.get_train_dataloader()))
        supervised_tokens = int(
            (batch["labels"] != -100).sum()
        )
        del batch

        assert supervised_tokens > 0, (
            "No supervised tokens. Check dataset formatting "
            "and MAX_SEQ_LEN."
        )

        print("Supervised tokens in first batch:", supervised_tokens)

        # Resume an existing checkpoint if present
        checkpoint_dir = Path(training_args.output_dir)

        checkpoint = (
            get_last_checkpoint(str(checkpoint_dir))
            if checkpoint_dir.exists() else None
        )

        if checkpoint:
            print("Resuming checkpoint:", checkpoint)

        # Start training
        local_trainer.train(
            resume_from_checkpoint=checkpoint
        )

        assert local_trainer.state.global_step > 0, (
            "Training completed zero optimizer steps"
        )

        # Save final trained adapter
        adapter_dir.mkdir(parents=True, exist_ok=True)

        local_trainer.model.save_pretrained(
            str(adapter_dir),
            safe_serialization=True,
        )
        tokenizer.save_pretrained(str(adapter_dir))

        assert saved_adapter_is_valid(adapter_dir), (
            f"Adapter save failed: {adapter_dir}"
        )

        # Save training history
        pd.DataFrame(
            local_trainer.state.log_history
        ).to_csv(
            OUT_DIR / f"training_log_{task_name}.csv",
            index=False,
        )

        print(f"\nSaved adapter: {adapter_dir}")
        print(
            "Completed steps:",
            local_trainer.state.global_step
        )

        return adapter_dir

    finally:
        # Release GPU memory before training next adapter
        del local_trainer, local_model
        gc.collect()
        torch.cuda.empty_cache()

In [14]:
# Attack detection first
train_one_adapter("attack_detection")


Training: attack_detection
Samples: 4998
Expected optimizer steps: 625
Warmup steps: 19


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

trainable params: 41,943,040 || all params: 7,289,966,592 || trainable%: 0.5754


Truncating train dataset:   0%|          | 0/4998 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/4998 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/475 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/475 [00:00<?, ? examples/s]

Supervised tokens in first batch: 3
Resuming checkpoint: /workspace/mistral_multi_lora/sai_split/trainer/attack_detection/checkpoint-200
{'loss': '0.07591', 'grad_norm': '0.04969', 'learning_rate': '0.0001076', 'entropy': '0.0429', 'mean_token_accuracy': '0.9783', 'num_tokens': '2.052e+05', 'epoch': '0.4802'}
{'loss': '0.04704', 'grad_norm': '0.408', 'learning_rate': '7.459e-05', 'entropy': '0.02165', 'mean_token_accuracy': '0.9892', 'num_tokens': '4.107e+05', 'epoch': '0.6403'}
{'eval_loss': '0.05564', 'eval_runtime': '59.31', 'eval_samples_per_second': '8.009', 'eval_steps_per_second': '8.009', 'eval_entropy': '0.07408', 'eval_mean_token_accuracy': '0.9804', 'eval_num_tokens': '4.107e+05', 'epoch': '0.6403'}
{'loss': '0.01944', 'grad_norm': '0.04259', 'learning_rate': '4.158e-05', 'entropy': '0.01612', 'mean_token_accuracy': '0.9963', 'num_tokens': '6.16e+05', 'epoch': '0.8003'}
{'loss': '0.01857', 'grad_norm': '0.002251', 'learning_rate': '8.581e-06', 'entropy': '0.01797', 'mean_tok

PosixPath('/workspace/mistral_multi_lora/sai_split/adapters/attack_detection')

In [15]:
train_one_adapter("tactic_classification")
train_one_adapter("technique_classification")


Training: tactic_classification
Samples: 4998
Expected optimizer steps: 625
Warmup steps: 19


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

trainable params: 41,943,040 || all params: 7,289,966,592 || trainable%: 0.5754


Truncating train dataset:   0%|          | 0/4998 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/4998 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/475 [00:00<?, ? examples/s]

... (6 outputs left out)

Truncating eval dataset:   0%|          | 0/475 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/475 [00:00<?, ? examples/s]

Supervised tokens in first batch: 13
{'loss': '1.735', 'grad_norm': '33.75', 'learning_rate': '0', 'entropy': '0.09367', 'mean_token_accuracy': '0.7435', 'num_tokens': '4594', 'epoch': '0.001601'}
{'loss': '0.2049', 'grad_norm': '0.1279', 'learning_rate': '0.0001736', 'entropy': '0.08834', 'mean_token_accuracy': '0.9553', 'num_tokens': '4.534e+05', 'epoch': '0.1601'}
{'loss': '0.06837', 'grad_norm': '0.00296', 'learning_rate': '0.0001406', 'entropy': '0.04455', 'mean_token_accuracy': '0.981', 'num_tokens': '9.073e+05', 'epoch': '0.3201'}
{'eval_loss': '0.05347', 'eval_runtime': '98.11', 'eval_samples_per_second': '4.841', 'eval_steps_per_second': '4.841', 'eval_entropy': '0.02463', 'eval_mean_token_accuracy': '0.9867', 'eval_num_tokens': '9.073e+05', 'epoch': '0.3201'}
{'loss': '0.05404', 'grad_norm': '0.03271', 'learning_rate': '0.0001076', 'entropy': '0.04478', 'mean_token_accuracy': '0.9838', 'num_tokens': '1.361e+06', 'epoch': '0.4802'}
{'loss': '0.05', 'grad_norm': '0.418', 'learn

PosixPath('/workspace/mistral_multi_lora/sai_split/adapters/technique_classification')

In [16]:
import gc
import torch
from pathlib import Path
from peft import PeftModel

# Verify all adapters exist
ADAPTER_ROOT = Path(ADAPTER_ROOT).resolve()

for task_name in TASKS:
    path = ADAPTER_ROOT / task_name
    assert saved_adapter_is_valid(path), (
        f"Missing adapter: {path}"
    )

# Release any previous model
if "model" in globals():
    del model

gc.collect()
torch.cuda.empty_cache()

# Load one shared Mistral base
model = load_base()

# Attach the first adapter
model = PeftModel.from_pretrained(
    model,
    str(ADAPTER_ROOT / "attack_detection"),
    adapter_name="attack_detection",
    is_trainable=False
)

# Attach remaining adapters
for task_name in TASKS[1:]:
    model.load_adapter(
        str(ADAPTER_ROOT / task_name),
        adapter_name=task_name,
        is_trainable=False
    )

# Configure inference
model.eval()
model.config.use_cache = True
model.generation_config.use_cache = True
tokenizer.padding_side = "left"

print("Loaded adapters:", list(model.peft_config.keys()))

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loaded adapters: ['attack_detection', 'tactic_classification', 'technique_classification']


In [17]:
sample = val_set.head(4)

for task_name in TASKS:
    model.set_adapter(task_name)

    outputs = infer_task(task_name, sample)

    print(f"\n{task_name}")

    for text, (parsed, probability, raw) in list(outputs.items())[:2]:
        print("Prediction:", parsed)
        if task_name == "attack_detection":
            print("P(attack):", round(probability, 4))
        print("Raw:", raw)

Inference: attack_detection:   0%|          | 0/1 [00:00<?, ?batch/s]


attack_detection
Prediction: {'attack': False, 'tactics': [], 'techniques': [], 'invalid': [], 'parsed': True}
P(attack): 0.0019
Raw: {"attack":false}
Prediction: {'attack': True, 'tactics': [], 'techniques': [], 'invalid': [], 'parsed': True}
P(attack): 1.0
Raw: {"attack":true}


Inference: tactic_classification:   0%|          | 0/1 [00:00<?, ?batch/s]


tactic_classification
Prediction: {'attack': None, 'tactics': [], 'techniques': [], 'invalid': [], 'parsed': True}
Raw: {"tactics": []}
Prediction: {'attack': None, 'tactics': ['Defense Evasion', 'Exfiltration'], 'techniques': [], 'invalid': [], 'parsed': True}
Raw: {"tactics": ["Defense Evasion", "Exfiltration"]}


Inference: technique_classification:   0%|          | 0/1 [00:00<?, ?batch/s]


technique_classification
Prediction: {'attack': None, 'tactics': [], 'techniques': [], 'invalid': [], 'parsed': True}
Raw: {"techniques": []}
Prediction: {'attack': None, 'tactics': [], 'techniques': ['T1048', 'T1078'], 'invalid': [], 'parsed': True}
Raw: {"techniques": ["T1048", "T1078"]}


In [18]:
import json
import time
import pandas as pd

tuned_scores_by_split = {}
tuned_preds_by_split = {}

for dataset_name, frame in test_sets.items():

    print(f"\nEvaluating: {dataset_name}")
    print(f"Connections: {len(frame):,}")

    start = time.perf_counter()

    pred = run_model(
        frame,
        label="fine_tuned",
        dataset_name=dataset_name
    )

    scores = score_predictions(pred, frame)

    tuned_preds_by_split[dataset_name] = pred
    tuned_scores_by_split[dataset_name] = scores

    # Save scores
    score_path = (
        OUT_DIR / f"scores_{dataset_name}_fine_tuned.json"
    )

    with open(score_path, "w") as f:
        json.dump(scores, f, indent=2)

    print(
        f"Elapsed: {(time.perf_counter() - start) / 60:.2f} min"
    )

    display(
        pd.Series(
            scores,
            name=f"{dataset_name} | fine-tuned"
        ).round(4)
    )


Evaluating: related_sessions
Connections: 7,882


Inference: attack_detection:   0%|          | 0/1032 [00:00<?, ?batch/s]

Inference: tactic_classification:   0%|          | 0/1032 [00:00<?, ?batch/s]

Inference: technique_classification:   0%|          | 0/1032 [00:00<?, ?batch/s]

Elapsed: 39.33 min


parsed_answers                 1.0000
answers_with_invalid_labels    0.0000
attack_pr_auc                  0.9902
attack_roc_auc                 0.9994
attack_recall                  0.9974
attack_precision               0.9176
false_alarm_rate               0.0080
tactics_exact_match            0.7561
tactics_first_correct          0.7700
tactics_micro_f1               0.7970
tactics_macro_f1               0.4817
techniques_exact_match         0.8017
techniques_first_correct       0.8105
techniques_micro_f1            0.8187
techniques_macro_f1            0.3593
Name: related_sessions | fine-tuned, dtype: float64

... (2 outputs left out)

Inference: tactic_classification:   0%|          | 0/224 [00:00<?, ?batch/s]

Inference: technique_classification:   0%|          | 0/224 [00:00<?, ?batch/s]

Elapsed: 12.04 min


parsed_answers                 1.0000
answers_with_invalid_labels    0.0000
attack_pr_auc                  0.1537
attack_roc_auc                 0.5819
attack_recall                  0.9685
attack_precision               0.1614
false_alarm_rate               0.8331
tactics_exact_match            0.0709
tactics_first_correct          0.2520
tactics_micro_f1               0.1869
tactics_macro_f1               0.0897
techniques_exact_match         0.0000
techniques_first_correct       0.1260
techniques_micro_f1            0.0860
techniques_macro_f1            0.0231
Name: unseen_synthetic | fine-tuned, dtype: float64

In [19]:
import json
import pandas as pd

comparison_frames = {}

for dataset_name in test_sets:

    zero_path = (
        OUT_DIR / f"scores_{dataset_name}_zero_shot.json"
    )

    with open(zero_path) as f:
        zero_scores = json.load(f)

    tuned_scores = tuned_scores_by_split[dataset_name]

    comparison_frames[dataset_name] = pd.DataFrame({
        "Zero-shot": zero_scores,
        "Multi-LoRA": tuned_scores
    })

    print(f"\n{dataset_name}")
    display(comparison_frames[dataset_name].round(4))

comparison = pd.concat(
    comparison_frames,
    names=["dataset", "metric"]
)

comparison.to_csv(
    OUT_DIR / "comparison_two_test_sets.csv"
)

print("Saved comparison results")


related_sessions


,Zero-shot,Multi-LoRA
parsed_answers,1.0000,1.0000
answers_with_invalid_labels,0.1290,0.0000
attack_pr_auc,0.0692,0.9902
attack_roc_auc,0.4321,0.9994
attack_recall,0.0012,0.9974
attack_precision,0.0050,0.9176
false_alarm_rate,0.0220,0.0080
tactics_exact_match,0.0000,0.7561
tactics_first_correct,0.0014,0.7700
tactics_micro_f1,0.3023,0.7970



unseen_synthetic


,Zero-shot,Multi-LoRA
parsed_answers,1.0000,1.0000
answers_with_invalid_labels,0.3680,0.0000
attack_pr_auc,0.1440,0.1537
attack_roc_auc,0.5264,0.5819
attack_recall,0.0079,0.9685
attack_precision,0.0455,0.1614
false_alarm_rate,0.0274,0.8331
tactics_exact_match,0.0315,0.0709
tactics_first_correct,0.1181,0.2520
tactics_micro_f1,0.2044,0.1869


Saved comparison results


In [20]:
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix

for name, frame in test_sets.items():
    pred = tuned_preds_by_split[name].loc[frame.index]

    y = frame["attack"].astype(int).to_numpy()
    yhat = pred["attack_pred"].astype(int).to_numpy()
    p = pred["p_attack"].to_numpy()

    w = (
        frame["weight"].to_numpy(dtype=float)
        if "weight" in frame
        else np.ones(len(frame))
    )

    tn, fp, fn, tp = confusion_matrix(
        y, yhat, labels=[0, 1],
        sample_weight=w
    ).ravel()

    print(f"\n{name}")
    print(f"TP={tp:.1f}, FP={fp:.1f}")
    print(f"FN={fn:.1f}, TN={tn:.1f}")

    print("True attack rate:",
          round(np.average(y, weights=w), 4))
    print("Predicted attack rate:",
          round(np.average(yhat, weights=w), 4))

    print("Disagreement between JSON and p>=0.5:",
          round(np.mean(yhat != (p >= 0.5)), 4))

    display(
        pd.DataFrame({
            "actual_attack": y,
            "p_attack": p
        }).groupby("actual_attack")["p_attack"].describe()
    )



related_sessions
TP=5655.2, FP=507.9
FN=14.8, TN=62983.1
True attack rate: 0.082
Predicted attack rate: 0.0891
Disagreement between JSON and p>=0.5: 0.0


,count,mean,std,min,25%,50%,75%,max
actual_attack,,,,,,,,
0,5000.0,0.011003,0.071392,2.699578e-07,0.000005,0.000031,0.000203,0.999877
1,2882.0,0.974789,0.067572,1.233946e-04,0.991938,0.998901,0.999979,1.000000



unseen_synthetic
TP=123.0, FP=639.0
FN=4.0, TN=128.0
True attack rate: 0.1421
Predicted attack rate: 0.8523
Disagreement between JSON and p>=0.5: 0.0


,count,mean,std,min,25%,50%,75%,max
actual_attack,,,,,,,,
0,767.0,0.844647,0.309908,0.008577,0.943348,0.999842,0.999942,0.999994
1,127.0,0.971447,0.158294,0.029312,0.999803,0.999877,0.999934,0.999992


In [21]:
train_texts = set(train_set["text"])

train_tactics = {
    x for labels in train_set["tactics"] for x in labels
}

train_techniques = {
    x for labels in train_set["techniques"] for x in labels
}

for name, frame in test_sets.items():

    test_tactics = {
        x for labels in frame["tactics"] for x in labels
    }

    test_techniques = {
        x for labels in frame["techniques"] for x in labels
    }

    print(f"\n{name}")

    print(
        "Exact text overlap:",
        f"{frame['text'].isin(train_texts).mean():.2%}"
    )

    print(
        "Tactics absent from training:",
        sorted(test_tactics - train_tactics)
    )

    print(
        "Techniques absent from training:",
        sorted(test_techniques - train_techniques)
    )



related_sessions
Exact text overlap: 53.95%
Tactics absent from training: []
Techniques absent from training: []

unseen_synthetic
Exact text overlap: 0.00%
Tactics absent from training: []
Techniques absent from training: []
